# CSF Data Analyst Internship Assessment 2026

**Name:** Nusrat Jahan  
**Role:** Data Analyst  
**Assessment:** Canadian Cheese Directory x Provincial Temperature Analysis

---

This notebook contains the full data analysis pipeline.
The interactive visualizations are served through a Dash web app.
Run the final cell and open your browser at **http://127.0.0.1:8050** to see the dashboard.

## 1. Import Libraries

In [1]:
# Importing the pandas library for reading and working with table data stored in CSV files
import pandas as pd

# Importing the numpy library for doing math operations like averages and number ranges
import numpy as np

# Importing the re library for reading and finding patterns inside text using regular expressions
import re

# Importing the warnings library and turning off unnecessary warning messages to keep the output clean
import warnings
warnings.filterwarnings('ignore')

# Printing a confirmation message to let the user know all libraries loaded without any errors
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Load Datasets

In [2]:
# Loading the cheese dataset from the nusratjdata folder into a pandas dataframe called cheese
# This file contains information about each cheese made in Canada including the province and milk type
cheese = pd.read_csv('nusratjdata/cheese_data.csv')

# Loading the weather dataset from the nusratjdata folder into a pandas dataframe called weather
# This file contains average temperature values for different cities across Canadian provinces
weather = pd.read_csv('nusratjdata/canada_weather.csv')

# Loading the large historical temperature dataset from the nusratjdata folder into a dataframe called temp
# This file contains monthly mean temperature readings from weather stations across Canada going back many decades
temp = pd.read_csv('nusratjdata/Canada_Temperature_Data.csv')

# Printing the number of rows and columns in each dataset so we know the size of the data we are working with
print(f'Cheese dataset:      {cheese.shape[0]:,} rows, {cheese.shape[1]} columns')
print(f'Weather dataset:     {weather.shape[0]:,} rows, {weather.shape[1]} columns')
print(f'Temperature dataset: {temp.shape[0]:,} rows, {temp.shape[1]} columns')

Cheese dataset:      1,042 rows, 13 columns
Weather dataset:     43 rows, 10 columns
Temperature dataset: 1,357,283 rows, 7 columns


## 3. Data Exploration

In [3]:
# Printing the first few rows of the cheese dataset to understand what the data looks like
print('=== CHEESE DATASET ===')
print(cheese.head())

# Printing the data type of each column so we know which columns hold text and which hold numbers
print('\nColumn info:')
print(cheese.dtypes)

# Counting and printing the number of missing values in each column to understand data quality
print('\nMissing values:')
print(cheese.isnull().sum())

=== CHEESE DATASET ===
   CheeseId ManufacturerProvCode ManufacturingTypeEn  MoisturePercent  \
0       228                   NB           Farmstead             47.0   
1       242                   NB           Farmstead             47.9   
2       301                   ON          Industrial             54.0   
3       303                   NB           Farmstead             47.0   
4       319                   NB           Farmstead             49.4   

                                          FlavourEn  \
0                                     Sharp, lactic   
1                Sharp, lactic, lightly caramelized   
2                           Mild, tangy, and fruity   
3  Sharp with fruity notes and a hint of wild honey   
4                                      Softer taste   

                                   CharacteristicsEn  Organic  \
0                                           Uncooked        0   
1                                           Uncooked        0   
2  Pressed a

In [4]:
# Printing the first few rows of the weather dataset to understand what columns and values it contains
print('=== WEATHER DATASET ===')
print(weather.head())

# Printing the data type of each column in the weather dataset to check if temperature values are stored as text or numbers
print('\nColumn info:')
print(weather.dtypes)

=== WEATHER DATASET ===
           Community Weather station  \
0       Alberton, PE             NaN   
1     Baker Lake, NU             YBK   
2    Baie-Comeau, QC             YBC   
3        Calgary, AB             YYC   
4  Charlottetown, PE             YYG   

                                            Location         Elevation  \
0  46°51′00″N 064°01′00″W / 46.85000°N 64.01667°W...        3m (9.8ft)   
1  64°17′56″N 096°04′40″W / 64.29889°N 96.07778°W...      18.6m (61ft)   
2  49°08′00″N 068°12′00″W / 49.13333°N 68.20000°W...        22m (72ft)   
3  51°06′50″N 114°01′13″W / 51.11389°N 114.02028°...  1,084m (3,556ft)   
4  46°17′19″N 063°07′43″W / 46.28861°N 63.12861°W...       49m (161ft)   

  January(Avg. high °C (°F)) January(Avg. low °C (°F))  \
0                −3.9 (25.0)               −12.5 (9.5)   
1              −27.7 (−17.9)             −34.8 (−30.6)   
2                −8.7 (16.3)              −19.9 (−3.8)   
3                −0.9 (30.4)               −13.2 (8.2)   


In [5]:
# Printing the first few rows of the temperature dataset to understand its structure and contents
print('=== TEMPERATURE DATASET ===')
print(temp.head())

# Printing all unique province codes found in the temperature dataset to see which provinces are included
print('\nUnique provinces:', temp['Prov'].unique())

# Printing the earliest and latest year in the dataset to understand the time range of the historical data
print('\nYear range:', temp['Year'].min(), 'to', temp['Year'].max())

=== TEMPERATURE DATASET ===
   Year  Month                   Stn_Name Prov   Tm     S      P
0  1917      1                   COWICHAN   BC  1.3  54.6  114.7
1  1917      1  COWICHAN BAY CHERRY POINT   BC  1.3  24.1   87.7
2  1917      1               JAMES ISLAND   BC  3.3  25.3   95.8
3  1917      1                  METCHOSIN   BC  2.2  46.9  156.4
4  1917      1             MILNES LANDING   BC  2.4  16.5  127.8

Unique provinces: ['BC' 'YT' 'NT' 'AB' 'SK' 'MB' 'ON' 'QC' 'NB' 'NS' 'PE' 'NL' 'NU' 'XX']

Year range: 1917 to 2017


## 4. Data Cleaning and Pipeline

## 4.0 Full Pipeline Overview

The diagram below shows the full end-to-end data pipeline used in this project.
Each stage feeds into the next, transforming three raw CSV files into a single clean analytical dataset ready for visualization.

```
┌────────────────────────────────────────────────────────────────────────────┐
│                        RAW DATA SOURCES                                    │
│                                                                            │
│  cheese_data.csv        canada_weather.csv    Canada_Temperature_Data.csv  │
│  1,042 rows             43 rows               1,357,283 rows               │
│  13 columns             10 columns             7 columns                   │
└────────────┬───────────────────┬──────────────────────┬────────────────────┘
             │                   │                      │
             ▼                   ▼                      ▼
┌─────────────────────────────────────────────────────────────────────┐
│                     STAGE 1: INGESTION                              │
│  Loading all three CSV files into pandas DataFrames                 │
└─────────────────────────────────────────────────────────────────────┘
             │
             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                     STAGE 2: CLEANING                               │
│  Cheese  → Remove nulls, filter to 10 provinces, fill unknowns      │
│  Weather → Regex extraction of Celsius from mixed text strings      │
│  Temp    → Filter provinces, remove null temperatures               │
└─────────────────────────────────────────────────────────────────────┘
             │
             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                  STAGE 3: FEATURE ENGINEERING                       │
│  Create Category column from long category names                    │
│  Create FatScore by mapping text fat levels to numeric values       │
│  Create ProvName full name lookup from province codes               │
│  Derive ColdMonthRatio from 1.3M rows of monthly temperature data   │
└─────────────────────────────────────────────────────────────────────┘
             │
             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                     STAGE 4: AGGREGATION                            │
│  Group by Province to compute:                                      │
│    CheeseCount, AvgMoisture, AvgFatScore, OrganicRatio              │
│    AvgMeanTemp, AvgJanHigh, AvgJulyHigh, UniqueMilkTypes            │
│    ColdMonthRatio, ShannonDiversityIndex                            │
└─────────────────────────────────────────────────────────────────────┘
             │
             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                     STAGE 5: MERGING                                │
│  Join cheese, weather and temperature tables on Province key        │
│  Result: one clean master DataFrame with all metrics per province   │
└─────────────────────────────────────────────────────────────────────┘
             │
             ▼
┌─────────────────────────────────────────────────────────────────────┐
│               STAGE 6: CREATIVE METRIC COMPUTATION                  │
│  Richness Score  = weighted formula combining 4 normalized metrics  │
│  Shannon Index   = milk type diversity entropy per province         │
│  Cold Month Ratio = proportion of months below 5°C per province     │
└─────────────────────────────────────────────────────────────────────┘
             │
             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                  STAGE 7: VISUALIZATION                             │
│  8 interactive Plotly charts served through a Dash web dashboard    │
└─────────────────────────────────────────────────────────────────────┘
```

### 4.1 Cleaning the Cheese Dataset

In [6]:
# Defining the list of ten valid Canadian provinces to keep in the analysis
valid_provinces = ['QC', 'ON', 'BC', 'AB', 'MB', 'SK', 'NS', 'NB', 'NL', 'PE']

# Defining full province names for display purposes in charts and tooltips
prov_names = {
    'QC': 'Quebec', 'ON': 'Ontario', 'BC': 'British Columbia',
    'AB': 'Alberta', 'MB': 'Manitoba', 'SK': 'Saskatchewan',
    'NS': 'Nova Scotia', 'NB': 'New Brunswick',
    'NL': 'Newfoundland', 'PE': 'Prince Edward Island'
}

# Removing all rows from the cheese dataset where the province code column is empty
# because we cannot link a cheese to any province without this information
cheese_clean = cheese.dropna(subset=['ManufacturerProvCode']).copy()

# Renaming the province column to a shorter and clearer name so it is easier to read and use later
cheese_clean.rename(columns={'ManufacturerProvCode': 'Province'}, inplace=True)

# Keeping only rows from the ten main Canadian provinces and removing territories
cheese_clean = cheese_clean[cheese_clean['Province'].isin(valid_provinces)]

# Filling missing values in text columns with the word Unknown so all rows remain usable
for col in ['CategoryTypeEn', 'MilkTypeEn', 'MilkTreatmentTypeEn', 'RindTypeEn', 'FatLevel']:
    cheese_clean[col] = cheese_clean[col].fillna('Unknown')

# Adding a readable full province name column to the cheese dataset
cheese_clean['ProvName'] = cheese_clean['Province'].map(prov_names)

# Printing the number of rows remaining after cleaning and a count of cheeses per province
print(f'Cheese after cleaning: {cheese_clean.shape[0]:,} rows')
print('\nCheese count by province:')
print(cheese_clean['Province'].value_counts())

Cheese after cleaning: 1,042 rows

Cheese count by province:
Province
QC    796
ON    115
BC     65
NB     27
AB     13
MB     11
NS     10
NL      2
PE      2
SK      1
Name: count, dtype: int64


### 4.2 Cleaning the Weather Dataset

In [7]:
# Defining a function that takes a temperature value written as text like 9.6 (49.3) and returns only the Celsius number
# The function also handles special minus signs that look different from a regular keyboard dash
def extract_celsius(val):
    # Returning not a number if the value is empty or missing
    if pd.isna(val):
        return np.nan
    # Replacing the unicode minus sign and en dash characters with a regular minus sign so Python can read the number correctly
    val = str(val).replace('\u2212', '-').replace('\u2013', '-')
    # Using a regular expression pattern to find the first number at the start of the text including any minus sign before it
    match = re.match(r'^\s*([-]?\d+\.?\d*)', val)
    # Returning the matched number as a float or returning not a number if no number was found
    return float(match.group(1)) if match else np.nan

# Extracting the two letter province code from the community name column using the part after the last comma
weather['Province']     = weather['Community'].str.extract(r',\s*([A-Z]{2})$')

# Applying the extract celsius function to each temperature column to convert all text values into usable numbers
weather['AnnualHigh_C'] = weather['Annual(Avg. high °C (°F))'].apply(extract_celsius)
weather['AnnualLow_C']  = weather['Annual(Avg. low °C (°F))'].apply(extract_celsius)
weather['JanHigh_C']    = weather['January(Avg. high °C (°F))'].apply(extract_celsius)
weather['JulyHigh_C']   = weather['July(Avg. high °C (°F))'].apply(extract_celsius)

# Grouping all cities by province and calculating the average of each temperature column
prov_weather = weather.groupby('Province').agg(
    AvgAnnualHigh=('AnnualHigh_C', 'mean'),
    AvgAnnualLow=('AnnualLow_C',  'mean'),
    AvgJanHigh=('JanHigh_C',      'mean'),
    AvgJulyHigh=('JulyHigh_C',    'mean')
).reset_index()

# Printing the resulting table showing one row of averaged temperature values for each province
print('Province level weather summary:')
print(prov_weather)

Province level weather summary:
   Province  AvgAnnualHigh  AvgAnnualLow  AvgJanHigh  AvgJulyHigh
0        AB       8.433333     -3.366667   -7.300000    23.100000
1        BC      12.240000      2.060000   -0.520000    24.600000
2        MB       3.266667     -7.500000  -17.166667    22.333333
3        NB      10.533333     -0.800000   -4.866667    24.966667
4        NL       6.566667     -1.433333   -5.666667    19.200000
5        NS      10.900000      2.166667   -0.533333    22.600000
6        NT      -1.300000    -10.466667  -22.200000    21.100000
7        NU      -7.925000    -15.325000  -25.575000    13.050000
8        ON      11.650000      1.450000   -4.550000    26.475000
9        PE       9.800000      1.400000   -3.500000    23.433333
10       QC       9.100000     -0.700000   -7.300000    24.066667
11       SK       7.933333     -4.066667  -10.933333    24.866667
12       YT       3.533333     -7.900000  -16.933333    22.166667


### 4.3 Cleaning the Temperature Dataset

In [8]:
# Keeping only the rows where the province code matches one of the ten main Canadian provinces
temp_clean = temp[temp['Prov'].isin(valid_provinces)].copy().dropna(subset=['Tm'])

# Grouping all temperature readings by province and calculating the overall average mean temperature
prov_temp = temp_clean.groupby('Prov')['Tm'].mean().reset_index()

# Renaming the columns so they are easier to understand when used in merging and charting steps
prov_temp.columns = ['Province', 'AvgMeanTemp_C']

# Sorting the provinces from warmest to coldest so the table is easy to read
prov_temp = prov_temp.sort_values('AvgMeanTemp_C', ascending=False)

# Printing the final table showing the average historical mean temperature for each province
print('Average mean temperature by province (historical):')
print(prov_temp)

Average mean temperature by province (historical):
  Province  AvgMeanTemp_C
1       BC       6.848785
5       NS       6.374093
6       ON       5.689031
7       PE       5.594824
3       NB       4.743894
0       AB       4.443528
4       NL       3.661499
8       QC       3.525982
9       SK       2.467776
2       MB       1.996501


### 4.4 Merging All Datasets

In [9]:
# Counting the total number of cheeses produced in each province
cheese_count = cheese_clean.groupby('Province').size().reset_index(name='CheeseCount')

# Calculating average moisture percentage per province
moisture = cheese_clean.groupby('Province')['MoisturePercent'].mean().reset_index()
moisture.columns = ['Province', 'AvgMoisture']

# Merging all provincial level data into one single dataframe for use in charts
merged = cheese_count \
    .merge(prov_temp, on='Province', how='left') \
    .merge(prov_weather[prov_weather['Province'].isin(valid_provinces)], on='Province', how='left') \
    .merge(moisture, on='Province', how='left')
merged['ProvName'] = merged['Province'].map(prov_names)

# Simplifying cheese category names for cleaner labels in the charts
cat_map = {
    'Firm Cheese': 'Firm', 'Semi-soft Cheese': 'Semi-soft',
    'Soft Cheese': 'Soft', 'Hard Cheese': 'Hard',
    'Fresh Cheese': 'Fresh', 'Veined Cheeses': 'Veined'
}
cheese_clean['Category'] = cheese_clean['CategoryTypeEn'].map(cat_map).fillna('Other')

# Preparing grouped dataframes for each chart type
cat_counts  = cheese_clean.groupby(['Province', 'Category']).size().reset_index(name='Count')
cat_counts['ProvName'] = cat_counts['Province'].map(prov_names)

milk_counts = cheese_clean.groupby(['Province', 'MilkTypeEn']).size().reset_index(name='Count')
milk_counts['ProvName'] = milk_counts['Province'].map(prov_names)

# Printing the final merged dataframe
print('Merged dataset:')
print(merged)

Merged dataset:
  Province  CheeseCount  AvgMeanTemp_C  AvgAnnualHigh  AvgAnnualLow  \
0       AB           13       4.443528       8.433333     -3.366667   
1       BC           65       6.848785      12.240000      2.060000   
2       MB           11       1.996501       3.266667     -7.500000   
3       NB           27       4.743894      10.533333     -0.800000   
4       NL            2       3.661499       6.566667     -1.433333   
5       NS           10       6.374093      10.900000      2.166667   
6       ON          115       5.689031      11.650000      1.450000   
7       PE            2       5.594824       9.800000      1.400000   
8       QC          796       3.525982       9.100000     -0.700000   
9       SK            1       2.467776       7.933333     -4.066667   

   AvgJanHigh  AvgJulyHigh  AvgMoisture              ProvName  
0   -7.300000    23.100000    42.346154               Alberta  
1   -0.520000    24.600000    41.490164      British Columbia  
2  -17.166

## 5. Data Quality Report


In [10]:
# Printing a before and after data quality report to show the effect of the cleaning pipeline
print('=' * 60)
print('DATA QUALITY REPORT')
print('=' * 60)

print('\n--- CHEESE DATASET ---')
print(f'Rows before cleaning : {cheese.shape[0]:,}')
cheese_temp = cheese.dropna(subset=['ManufacturerProvCode']).copy()
cheese_temp.rename(columns={'ManufacturerProvCode': 'Province'}, inplace=True)
cheese_temp = cheese_temp[cheese_temp['Province'].isin(valid_provinces)]
print(f'Rows after cleaning  : {cheese_temp.shape[0]:,}')
print(f'Rows removed         : {cheese.shape[0] - cheese_temp.shape[0]:,}')
print(f'Missing province rows: {cheese["ManufacturerProvCode"].isna().sum()}')
print(f'Provinces kept       : {sorted(cheese_temp["Province"].unique().tolist())}')

print('\n--- WEATHER DATASET ---')
print(f'Rows total           : {weather.shape[0]:,}')
print(f'Temperature columns  : stored as mixed text like 9.6 (49.3) converted to float')
print(f'Provinces extracted  : {sorted(weather["Province"].dropna().unique().tolist())}')

print('\n--- TEMPERATURE DATASET ---')
print(f'Rows before cleaning : {temp.shape[0]:,}')
temp_q = temp[temp["Prov"].isin(valid_provinces)].dropna(subset=["Tm"])
print(f'Rows after cleaning  : {temp_q.shape[0]:,}')
print(f'Rows removed         : {temp.shape[0] - temp_q.shape[0]:,}')
print(f'Year range           : {temp_q["Year"].min()} to {temp_q["Year"].max()}')
print(f'Provinces covered    : {sorted(temp_q["Prov"].unique().tolist())}')

print('\n--- MERGED MASTER DATASET ---')
print(f'Final rows           : {merged.shape[0]} (one row per province)')
print(f'Final columns        : {merged.shape[1]}')
print(f'Columns              : {merged.columns.tolist()}')
print('\nSample of merged dataset:')
print(merged[['Province','CheeseCount','AvgMeanTemp_C','AvgMoisture']].to_string(index=False))
print('\n' + '=' * 60)

DATA QUALITY REPORT

--- CHEESE DATASET ---
Rows before cleaning : 1,042
Rows after cleaning  : 1,042
Rows removed         : 0
Missing province rows: 0
Provinces kept       : ['AB', 'BC', 'MB', 'NB', 'NL', 'NS', 'ON', 'PE', 'QC', 'SK']

--- WEATHER DATASET ---
Rows total           : 43
Temperature columns  : stored as mixed text like 9.6 (49.3) converted to float
Provinces extracted  : ['AB', 'BC', 'MB', 'NB', 'NL', 'NS', 'NT', 'NU', 'ON', 'PE', 'QC', 'SK', 'YT']

--- TEMPERATURE DATASET ---
Rows before cleaning : 1,357,283
Rows after cleaning  : 1,286,984
Rows removed         : 70,299
Year range           : 1917 to 2017
Provinces covered    : ['AB', 'BC', 'MB', 'NB', 'NL', 'NS', 'ON', 'PE', 'QC', 'SK']

--- MERGED MASTER DATASET ---
Final rows           : 10 (one row per province)
Final columns        : 9
Columns              : ['Province', 'CheeseCount', 'AvgMeanTemp_C', 'AvgAnnualHigh', 'AvgAnnualLow', 'AvgJanHigh', 'AvgJulyHigh', 'AvgMoisture', 'ProvName']

Sample of merged dataset

In [11]:
# Grouping the cheese data by province and organic status and counting how many cheeses fall into each group
organic_by_prov = cheese_clean.groupby(['Province','Organic']).size().unstack(fill_value=0)
organic_by_prov.columns = ['Non Organic', 'Organic']
print('Organic vs Non Organic cheese by province:')
print(organic_by_prov)

# Counting how many cheeses are made from each type of milk
print('\nMilk type distribution:')
print(cheese_clean['MilkTypeEn'].value_counts())

# Calculating the average moisture percentage of cheeses in each province
print('\nAverage moisture percentage by province:')
print(cheese_clean.groupby('Province')['MoisturePercent'].mean().sort_values(ascending=False))

Organic vs Non Organic cheese by province:
          Non Organic  Organic
Province                      
AB                 13        0
BC                 45       20
MB                 10        1
NB                 24        3
NL                  2        0
NS                  9        1
ON                113        2
PE                  2        0
QC                724       72
SK                  1        0

Milk type distribution:
MilkTypeEn
Cow                  743
Goat                 214
Ewe                   62
Cow and Goat          13
Ewe and Cow            4
Ewe and Goat           2
Buffalo Cow            2
Cow, Goat and Ewe      1
Unknown                1
Name: count, dtype: int64

Average moisture percentage by province:
Province
NB    49.603704
QC    47.715044
ON    47.085455
AB    42.346154
MB    41.545455
BC    41.490164
NS    41.300000
NL    39.500000
PE    39.500000
SK    17.000000
Name: MoisturePercent, dtype: float64


## 6. Creative Metrics and Feature Engineering

Beyond standard analysis, this project engineers four custom metrics to uncover deeper patterns in the data.

---

### Metric 1: Cheese Style Heatmap
**What it shows:** A grid where rows are cheese categories (Firm, Soft, Fresh, etc.) and columns are provinces ordered from coldest to warmest. The cell color intensity shows how many cheeses of that style each province produces.

**Why it is interesting:** It makes the relationship between climate and cheese style immediately visible. Cold provinces like Quebec and Manitoba dominate Firm and Semi-soft categories, while warmer provinces like BC show stronger Soft cheese production.

---

### Metric 2: Cheese Richness Score (Custom Engineered Metric)
**Formula:**
```
RichnessScore = (Moisture × 0.25) + (FatScore × 0.25) + (MilkDiversity × 0.30) + (OrganicRatio × 0.20)
```
All components are normalized to 0-1 before combining. The final score is scaled to 0-100.

**What each component means:**
- Moisture (25%) — wetter cheeses tend to be richer and more complex
- Fat Score (25%) — higher fat content indicates richer cheese styles
- Milk Diversity (30%) — provinces using more milk types produce a wider variety
- Organic Ratio (20%) — higher organic production signals artisan quality focus

**Why it is interesting:** Quebec scores highest despite being the coldest, showing that cold climate encourages artisan richness rather than mass production simplicity.

---

### Metric 3: Cold Months vs Cheese Production
**What it shows:** Using all 1,357,283 rows of historical temperature data, this metric calculates the proportion of months per year where mean temperature falls below 5°C for each province. This is then plotted against cheese production count.

**Why it is interesting:** It transforms a massive raw dataset into a single meaningful number per province and reveals that provinces spending more months in cold conditions tend to produce more cheese — supporting the cultural hypothesis that cold-climate communities historically developed stronger cheese-making traditions.

---

### Metric 4: Shannon Milk Type Diversity Index
**Formula:**
```
H = -Σ (p_i × ln(p_i))
```
Where p_i is the proportion of cheeses made from each milk type in a province.

**What it shows:** A higher Shannon Index means a province uses many different milk types in balanced proportions. A score near zero means a province relies almost entirely on one milk type.

**Why it is interesting:** Ontario and Alberta have the highest diversity scores despite not being the largest producers, suggesting these provinces have more experimental or varied dairy farming cultures compared to Quebec which dominates volume but relies heavily on cow milk.

In [12]:
# ── CREATIVE METRIC 1: CHEESE RICHNESS SCORE ────────────────────────────────

# Mapping fat level text labels to numeric scores so they can be included in the richness formula
fat_map = {'lower fat': 1, 'light': 2, 'medium': 3, 'high': 4, 'extra high': 5}
cheese_clean['FatScore'] = cheese_clean['FatLevel'].str.lower().map(fat_map).fillna(3)

# Aggregating all four ingredients of the richness score per province
richness_raw = cheese_clean.groupby('Province').agg(
    AvgMoisture=('MoisturePercent', 'mean'),
    AvgFat=('FatScore', 'mean'),
    OrganicRatio=('Organic', 'mean'),
    UniqueMilkTypes=('MilkTypeEn', 'nunique')
).reset_index()

# Normalizing each component to a 0 to 1 scale before combining into the final weighted score
def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx != mn else pd.Series([0.5]*len(s), index=s.index)

richness_raw['RichnessScore'] = (
    norm(richness_raw['AvgMoisture'])       * 25 +
    norm(richness_raw['AvgFat'])            * 25 +
    norm(richness_raw['UniqueMilkTypes'])   * 30 +
    richness_raw['OrganicRatio']            * 20
)
richness_raw['ProvName'] = richness_raw['Province'].map(prov_names)
print('Cheese Richness Score by Province:')
print(richness_raw[['Province','ProvName','RichnessScore']].sort_values('RichnessScore', ascending=False).to_string(index=False))

# ── CREATIVE METRIC 2: COLD MONTH RATIO ─────────────────────────────────────

# Calculating the proportion of months where temperature is below 5 degrees Celsius per province
# Using all 1,357,283 rows of the historical temperature dataset for this calculation
cold_ratio = (
    temp_clean[temp_clean['Tm'] < 5].groupby('Prov').size() /
    temp_clean.groupby('Prov').size()
).reset_index()
cold_ratio.columns = ['Province', 'ColdMonthRatio']
cold_ratio['ColdMonths'] = (cold_ratio['ColdMonthRatio'] * 12).round(1)
cold_ratio['ProvName']   = cold_ratio['Province'].map(prov_names)
cold_merged = cold_ratio.merge(cheese_count, on='Province', how='left')
cold_merged['ProvName'] = cold_merged['Province'].map(prov_names)
cold_merged = cold_merged.merge(prov_temp, on='Province', how='left')
print('\nCold Months per Year by Province:')
print(cold_merged[['Province','ColdMonths','CheeseCount']].sort_values('ColdMonths', ascending=False).to_string(index=False))

# ── CREATIVE METRIC 3: SHANNON DIVERSITY INDEX ──────────────────────────────

# Calculating Shannon entropy for milk type distribution within each province
milk_valid   = cheese_clean[cheese_clean['MilkTypeEn'] != 'Unknown'].copy()
milk_grouped = milk_valid.groupby(['Province', 'MilkTypeEn']).size().reset_index(name='Count')

def shannon_index(group):
    # Computing entropy which is higher when many milk types are used in equal proportions
    total = group['Count'].sum()
    p = group['Count'] / total
    return -np.sum(p * np.log(p + 1e-12))

diversity = milk_grouped.groupby('Province').apply(shannon_index).reset_index()
diversity.columns = ['Province', 'ShannonIndex']
diversity['ProvName'] = diversity['Province'].map(prov_names)
diversity = diversity.merge(cheese_count, on='Province', how='left')
diversity = diversity.sort_values('ShannonIndex', ascending=True)
print('\nShannon Milk Type Diversity Index by Province:')
print(diversity[['Province','ProvName','ShannonIndex']].sort_values('ShannonIndex', ascending=False).to_string(index=False))

Cheese Richness Score by Province:
Province             ProvName  RichnessScore
      QC               Quebec      63.683716
      BC     British Columbia      49.273173
      AB              Alberta      48.116345
      ON              Ontario      44.317494
      NS          Nova Scotia      37.418567
      NB        New Brunswick      35.793651
      NL         Newfoundland      29.752641
      PE Prince Edward Island      29.752641
      SK         Saskatchewan      25.000000
      MB             Manitoba      22.911972

Cold Months per Year by Province:
Province  ColdMonths  CheeseCount
      NL         6.4            2
      MB         6.1           11
      QC         6.1          796
      SK         6.1            1
      NB         5.9           27
      PE         5.9            2
      NS         5.5           10
      ON         5.4          115
      AB         5.1           13
      BC         4.6           65

Shannon Milk Type Diversity Index by Province:
Province     

## 7. Interactive Dashboard

Running the cell below will start the Dash dashboard.  
Once it says **Dash is running on http://127.0.0.1:8050** open that link in your browser to see the interactive charts.

In [ ]:
# Importing the threading and webbrowser libraries to open the dashboard in a new browser tab automatically
import threading
import webbrowser

# Importing the subprocess library to run the dashboard script as a completely separate background process
import subprocess
import sys
import time

# Printing a clear message so the user knows where to find the dashboard
print('Starting the interactive dashboard...')
print('Opening your browser automatically at http://127.0.0.1:8050')
print('If it does not open, click this link: http://127.0.0.1:8050')

# Defining a function that waits 2 seconds then opens the browser so Dash has time to start first
def open_browser():
    time.sleep(2)
    webbrowser.open('http://127.0.0.1:8050')

# Starting the browser opener in a background thread so it does not block the notebook
threading.Thread(target=open_browser, daemon=True).start()

# Running the dashboard script as a subprocess so it does not render inside the notebook
subprocess.run([sys.executable, 'nusratj_dashboard.py'])

Starting the interactive dashboard...
Opening your browser automatically at http://127.0.0.1:8050
If it does not open, click this link: http://127.0.0.1:8050


## 8. Discussion

### Are there any inferences you can make about the relationship between the weather in a province and the cheese produced?

The analysis reveals a striking geographic concentration in Canadian cheese production: Quebec accounts for over 76% of all cheeses in the directory, followed distantly by Ontario (11%) and British Columbia (6%). When cross-referenced with historical temperature data, an interesting pattern emerges where the highest producing provinces tend to cluster in the moderate to cold temperature range, with Quebec averaging approximately 3.5 degrees Celsius mean annual temperature, Ontario around 5.7 degrees Celsius, and British Columbia sitting warmer at 6.8 degrees Celsius.

This suggests that colder climates may actually support or at least correlate with a higher diversity and volume of cheese production. Colder temperatures are historically associated with strong pastoral farming traditions, where dairy herds are well suited to cooler grasslands. Quebec's dominance in particular reflects its deep rooted French Canadian agricultural heritage, where cheese making has been a cultural practice for centuries, shaped by its cold continental climate. The cold winters and moderate summers of Quebec and Ontario create conditions well suited to the slow aging and fermentation processes that define many of the firm, semi soft, and washed rind cheeses found in those provinces.

In contrast, warmer or more extreme climate provinces like Manitoba and Saskatchewan produce very few cheeses despite having significant dairy industries. This may suggest that while temperature alone does not determine cheese production, the combination of climate, agricultural tradition, and population density plays a role, which are factors that Quebec and Ontario benefit from disproportionately.

It is also notable that British Columbia, the warmest of the higher producing provinces, tends to produce more soft and fresh cheese varieties compared to the firmer aged varieties dominant in Quebec. This aligns with the idea that warmer climates favor faster ripening higher moisture cheeses, while colder regions are better suited to the slow controlled aging environments needed for firm and hard cheeses.

Overall, while climate is not the sole determinant of cheese production patterns in Canada, the data suggests that moderate to cold provinces with strong dairy farming traditions produce the most diverse and voluminous cheese outputs, and that climate subtly shapes the style of cheese produced in each region.

Submitted by Nusrat Jahan, Data Analyst Internship Assessment, Canadian Sheep Federation, 2026